# 📡 Time-to-Depth Conversion (Seismic)

## 🔷 Overview
Seismic data is recorded in **time domain (TWT in ms)**, but interpretation requires **depth (meters)**.  
Time-to-depth conversion uses a **velocity cube** to transform seismic data into depth.

---

## 🔷 Basic Concept

Seismic waves travel down and back → recorded as **Two-Way Travel Time (TWT)**.

Depth is computed using velocity:

Z = ∫ (V × dt) / 2

- V = velocity (m/s)  
- dt = time interval  
- 1/2 → because of two-way travel  

---

## 🔷 Discrete Form (Used in Code)

Z[i] = Z[i-1] + (V[i] × Δt) / 2

- Δt = sampling interval  
- V[i] = velocity at sample i  

---

## 🔷 Workflow

1. Load seismic data (time domain)  
2. Load velocity cube  
3. Match grids (time samples must align)  
4. Compute depth using cumulative integration  
5. Interpolate to uniform depth grid  
6. Visualize depth section  

---

## 🔷 Key Points

- Use **interval velocity** (not RMS directly)  
- Velocity errors → depth errors  
- Same sampling between seismic & velocity is important  

---

## 🔷 Application

- Depth structure mapping  
- Well planning  
- Reservoir thickness estimation  

---


In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import segyio

In [2]:
seismic_time_data="/home/ashraf/Downloads/ST8511r92.segy"
velocity_data="/media/ashraf/𝓐𝓢𝓗𝓡𝓐𝓕1/Average_Velocity_cube.segy"

In [9]:
import segyio
import numpy as np

def load_segy(filepath):
    with segyio.open(filepath, "r", ignore_geometry=True) as f:
        f.mmap()

        try:
            # Try cube (for structured data)
            data = segyio.tools.cube(f)   # (iline, xline, time)
            data = data.reshape(-1, data.shape[-1]).T
            print("Loaded as cube (structured data)")
        
        except:
            # Fallback (unstructured)
            data = np.array([trace for trace in f.trace[:]]).T
            print("Loaded as raw traces (no geometry)")

        dt = segyio.tools.dt(f) / 1e6  # sec

    return data, dt


# Load velocity cube
velocity, dt = load_segy("/media/ashraf/𝓐𝓢𝓗𝓡𝓐𝓕1/Average_Velocity_cube.segy")

print("Shape:", velocity.shape)
print("dt:", dt)

Loaded as raw traces (no geometry)
Shape: (44, 823680)
dt: 0.004


In [10]:
# seismic, dt = load_segy("seismic.sgy")
seismic,dt= load_segy("/home/ashraf/Downloads/ST8511r92.segy")

print("Seismic shape:", seismic.shape)
print("dt:")

Loaded as raw traces (no geometry)
Seismic shape: (227, 112875)
dt:


In [ ]:
filepath="/home/ashraf/Downloads/ST8511r92.segy"
with segyio.open(filepath, "r", ignore_geometry=True) as f:
    print(f.header[0])

RuntimeError: unable to find sorting.Check iline, (189) and xline (193) in case you are sure the file is a 3D sorted volume